In [16]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')

print("STOCK LEDGER")
print("Rows:", len(ledger))
print("Columns:", list(ledger.columns))
print(ledger.head(5).to_string())

print()
print("INVENTORY MASTER")
print("Rows:", len(inv))
print("Columns:", list(inv.columns))
print(inv.head(5).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STOCK LEDGER
Rows: 237230
Columns: ['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']
      movement_id product_id branch_id movement_type movement_date  quantity reference_type reference_id  running_balance
0  MOV-OUT-184742       P001    AHM001           OUT    2019-01-03         8             SO    SO-157767              228
1  MOV-OUT-208422       P001    AHM001           OUT    2019-01-05        12             SO    SO-168735              216
2   MOV-IN-111128       P001    AHM001            IN    2019-01-23       174             PO    PO-181094              390
3  MOV-OUT-159140       P001    AHM001           OUT    2019-01-25        12             SO    SO-947377              378
4  MOV-OUT-173693       P001    AHM001           OUT    2019-01-25        11        

In [17]:
import numpy as np

first = ledger.groupby(['product_id', 'branch_id'], as_index=False).first()

first['balance_before'] = np.where(
    first['movement_type'] == 'IN', first['running_balance'] - first['quantity'],
    np.where(first['movement_type'] == 'OUT', first['running_balance'] + first['quantity'], np.nan))

merged = first.merge(inv[['product_id', 'branch_id', 'opening_stock']],
                     on=['product_id', 'branch_id'], how='outer', indicator=True)

print("Pairs found in ledger and inventory (both):", (merged['_merge'] == 'both').sum())
print("Pairs only in one table:", (merged['_merge'] != 'both').sum())
print()
print("Type of each pair's first ledger row:")
print(merged['movement_type'].value_counts())

testable = merged[merged['balance_before'].notna()]
same = testable['balance_before'] == testable['opening_stock']
print()
print("Pairs we can test:", len(testable))
print("Balance before first row = opening_stock:", same.sum())
print("Does NOT match:", (~same).sum())

print()
print(testable.loc[~same, ['product_id', 'branch_id', 'movement_type', 'balance_before', 'opening_stock']].head(10).to_string())

Pairs found in ledger and inventory (both): 180
Pairs only in one table: 0

Type of each pair's first ledger row:
movement_type
OUT           146
IN             21
ADJUSTMENT     13
Name: count, dtype: int64

Pairs we can test: 167
Balance before first row = opening_stock: 167
Does NOT match: 0

Empty DataFrame
Columns: [product_id, branch_id, movement_type, balance_before, opening_stock]
Index: []


In [18]:
adj_first = merged[merged['movement_type'] == 'ADJUSTMENT'].copy()

adj_first['change'] = adj_first['running_balance'] - adj_first['opening_stock']

adj_first['direction'] = np.select(
    [adj_first['change'] == adj_first['quantity'],
     adj_first['change'] == -adj_first['quantity']],
    ['adds', 'subtracts'],
    default='unreadable')

print("Pairs that start with an ADJUSTMENT:", len(adj_first))
print()
print(adj_first[['product_id', 'branch_id', 'quantity', 'opening_stock',
                 'running_balance', 'change', 'direction']].to_string())
print()
print(adj_first['direction'].value_counts())

Pairs that start with an ADJUSTMENT: 13

    product_id branch_id  quantity  opening_stock  running_balance  change  direction
16        P003    KOL001        19            234              253      19       adds
47        P008    PUN001        43             86               43     -43  subtracts
51        P009    HYD001        40            282              322      40       adds
72        P013    AHM001        28             84              112      28       adds
75        P013    HYD001        37            233              270      37       adds
86        P015    DEL001        37            223              186     -37  subtracts
91        P016    CHN001        42            180              222      42       adds
94        P016    KOL001        33            148              181      33       adds
150       P026    AHM001        38            263              301      38       adds
157       P027    CHN001        29             84               55     -29  subtracts
158       P02

In [19]:
led = ledger.copy()

led['prev_balance'] = led.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
led = led.merge(inv[['product_id', 'branch_id', 'opening_stock']],
                on=['product_id', 'branch_id'], how='left')
led['prev_balance'] = led['prev_balance'].fillna(led['opening_stock'])
led['change'] = led['running_balance'] - led['prev_balance']

is_in = led['movement_type'] == 'IN'
is_out = led['movement_type'] == 'OUT'
is_adj = led['movement_type'] == 'ADJUSTMENT'

print("Total ledger rows:", len(led))
print()
print("IN rows:", is_in.sum(), "| change = +quantity:",
      (led.loc[is_in, 'change'] == led.loc[is_in, 'quantity']).sum())
print("OUT rows:", is_out.sum(), "| change = -quantity:",
      (led.loc[is_out, 'change'] == -led.loc[is_out, 'quantity']).sum())

adj = led[is_adj].copy()
adj['direction'] = np.select(
    [adj['change'] == adj['quantity'], adj['change'] == -adj['quantity']],
    ['adds', 'subtracts'], default='unreadable')
print()
print("ADJUSTMENT rows:", len(adj))
print(adj['direction'].value_counts())

Total ledger rows: 237230

IN rows: 127611 | change = +quantity: 127611
OUT rows: 107165 | change = -quantity: 107165

ADJUSTMENT rows: 2454
direction
subtracts    1227
adds         1227
Name: count, dtype: int64


In [20]:
is_in = led['movement_type'] == 'IN'
is_out = led['movement_type'] == 'OUT'
is_adj = led['movement_type'] == 'ADJUSTMENT'

led['signed_qty'] = np.select(
    [is_in, is_out, is_adj & (led['change'] == led['quantity']), is_adj & (led['change'] == -led['quantity'])],
    [led['quantity'], -led['quantity'], led['quantity'], -led['quantity']],
    default=0)

rebuilt = led.groupby(['product_id', 'branch_id']).agg(
    signed_total=('signed_qty', 'sum'),
    last_balance=('running_balance', 'last'),
    lowest_balance=('running_balance', 'min')).reset_index()

rebuilt = rebuilt.merge(inv[['product_id', 'branch_id', 'opening_stock', 'current_stock']],
                        on=['product_id', 'branch_id'], how='left')
rebuilt['reconstructed_stock'] = rebuilt['opening_stock'] + rebuilt['signed_total']

print("Pairs:", len(rebuilt))
print("Rows with signed_qty = 0:", (led['signed_qty'] == 0).sum())
print("Rebuilt stock = last ledger balance:", (rebuilt['reconstructed_stock'] == rebuilt['last_balance']).sum())
print("Pairs that ever went below zero:", (rebuilt['lowest_balance'] < 0).sum())
print("current_stock = rebuilt stock:", (rebuilt['current_stock'] == rebuilt['reconstructed_stock']).sum())
print()
print(rebuilt['reconstructed_stock'].describe())

Pairs: 180
Rows with signed_qty = 0: 0
Rebuilt stock = last ledger balance: 180
Pairs that ever went below zero: 0
current_stock = rebuilt stock: 166

count       180.000000
mean     107255.522222
std        5348.566182
min       88853.000000
25%      103487.500000
50%      107790.500000
75%      110779.500000
max      121013.000000
Name: reconstructed_stock, dtype: float64


In [21]:
diff = rebuilt[rebuilt['current_stock'] != rebuilt['reconstructed_stock']].copy()
diff['difference'] = diff['current_stock'] - diff['reconstructed_stock']
print("Pairs where current_stock differs from rebuilt stock:", len(diff))
print(diff[['product_id', 'branch_id', 'opening_stock', 'reconstructed_stock',
            'current_stock', 'difference']].to_string())

print()
print("Total units by movement type:")
print(led.groupby('movement_type')['quantity'].sum())
net_adj = led.loc[is_adj, 'signed_qty'].sum()
print("Net effect of adjustments:", net_adj)

print()
total_in = led.loc[is_in, 'quantity'].sum()
total_out = led.loc[is_out, 'quantity'].sum()
print("Total opening stock:", inv['opening_stock'].sum())
print("Opening + IN - OUT + adjustments:", inv['opening_stock'].sum() + total_in - total_out + net_adj)
print("Total rebuilt stock:", rebuilt['reconstructed_stock'].sum())
print()
print("Average max_stock:", inv['max_stock'].mean())

Pairs where current_stock differs from rebuilt stock: 14
    product_id branch_id  opening_stock  reconstructed_stock  current_stock  difference
7         P002    CHN001            267               106182         105885        -297
23        P004    PUN001            171               105024         104859        -165
42        P008    AHM001            259               109317         109281         -36
54        P010    AHM001            128               115549         115149        -400
122       P021    DEL001            152               116107         115902        -205
135       P023    HYD001             82               105981         105798        -183
147       P025    HYD001             83               114128         113613        -515
153       P026    HYD001            203               108546         108348        -198
155       P026    PUN001            112               105335         105190        -145
164       P028    DEL001            207               106967   

In [22]:
rows = []
for _, r in diff.iterrows():
    sub = led[(led['product_id'] == r['product_id']) & (led['branch_id'] == r['branch_id'])].reset_index(drop=True)
    hits = sub.index[sub['running_balance'] == r['current_stock']]
    if len(hits) > 0:
        k = hits[-1]
        rows.append([r['product_id'], r['branch_id'], r['current_stock'], 'yes',
                     sub.loc[k, 'movement_date'], len(sub) - 1 - k, sub['movement_date'].iloc[-1]])
    else:
        rows.append([r['product_id'], r['branch_id'], r['current_stock'], 'no',
                     None, None, sub['movement_date'].iloc[-1]])

check = pd.DataFrame(rows, columns=['product_id', 'branch_id', 'current_stock',
                                    'found_in_ledger', 'date_of_that_row',
                                    'rows_after_it', 'last_ledger_date'])
print(check.to_string())
print()
print(check['found_in_ledger'].value_counts())

   product_id branch_id  current_stock found_in_ledger date_of_that_row  rows_after_it last_ledger_date
0        P002    CHN001         105885             yes       2025-01-20              1       2025-01-20
1        P004    PUN001         104859             yes       2025-01-18              1       2025-01-18
2        P008    AHM001         109281             yes       2025-01-24              1       2025-01-24
3        P010    AHM001         115149             yes       2025-01-24              2       2025-01-24
4        P021    DEL001         115902             yes       2025-01-17              1       2025-01-17
5        P023    HYD001         105798             yes       2025-01-15              1       2025-01-15
6        P025    HYD001         113613             yes       2025-01-13              2       2025-01-13
7        P026    HYD001         108348             yes       2025-01-18              1       2025-01-18
8        P026    PUN001         105190             yes       202

In [23]:
out = []
for _, r in diff.iterrows():
    sub = led[(led['product_id'] == r['product_id']) & (led['branch_id'] == r['branch_id'])].reset_index(drop=True)
    k = sub.index[sub['running_balance'] == r['current_stock']][-1]
    tail = sub.loc[k + 1:, ['product_id', 'branch_id', 'movement_id', 'movement_type',
                            'movement_date', 'quantity', 'reference_type',
                            'reference_id', 'running_balance']]
    out.append(tail)

last_rows = pd.concat(out)
print("Rows after the match:", len(last_rows))
print(last_rows.to_string())
print()
print(last_rows['movement_type'].value_counts())
print()
print(last_rows['reference_type'].value_counts())

Rows after the match: 17
     product_id branch_id     movement_id movement_type movement_date  quantity reference_type reference_id  running_balance
1461       P002    CHN001   MOV-IN-114717            IN    2025-01-20       297             PO    PO-148144           106182
1241       P004    PUN001    MOV-IN-63458            IN    2025-01-18       165             PO    PO-541972           105024
1349       P008    AHM001   MOV-IN-135756            IN    2025-01-24        36             PO    PO-625015           109317
1398       P010    AHM001    MOV-IN-99729            IN    2025-01-24       142             PO    PO-327036           115291
1399       P010    AHM001   MOV-IN-135755            IN    2025-01-24       258             PO    PO-625015           115549
1257       P021    DEL001   MOV-IN-114633            IN    2025-01-17       205             PO    PO-167534           116107
1133       P023    HYD001   MOV-IN-105362            IN    2025-01-15       183             PO    PO

In [24]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

files = sorted(os.listdir(BASE_PATH))
print("Files in the folder:")
for f in files:
    print(" ", f)

print()
for f in files:
    name = f.lower()
    if f.endswith('.csv') and ('order' in name or 'purchase' in name or 'sales' in name):
        t = pd.read_csv(BASE_PATH + f)
        print(f, "| rows:", len(t))
        print("Columns:", list(t.columns))
        print(t.head(2).to_string())
        print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in the folder:
  branches.csv
  customers.csv
  inventory_master.csv
  invoices.csv
  payments.csv
  products.csv
  purchase_orders_header.csv
  purchase_orders_lines.csv
  sales_orders_header.csv
  sales_orders_lines.csv
  stock_ledger.csv
  suppliers.csv

purchase_orders_header.csv | rows: 24000
Columns: ['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
       po_id supplier_id branch_id  order_date expected_delivery_date received_date po_status   total_cost  total_gst_amount   grand_total
0  PO-141186     SUP0001    PUN001  2019-04-10             2019-04-29    2019-04-29  Received  22955232.11      6.427465e+06  2.938270e+07
1  PO-215234     SUP0005    DEL001  2020-12-30             2021-01-27    2021-01-27  Received  19406544.39      5.428358e+

In [25]:
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')

po_qty = po_l.groupby(['po_id', 'product_id'], as_index=False)['quantity'].sum()
po_qty = po_qty.rename(columns={'quantity': 'order_line_qty'})

po_rows = last_rows[last_rows['reference_type'] == 'PO'].copy()
po_rows = po_rows.merge(po_h[['po_id', 'branch_id', 'order_date', 'received_date', 'po_status']],
                        left_on='reference_id', right_on='po_id', how='left',
                        suffixes=('', '_order'))
po_rows = po_rows.merge(po_qty, on=['po_id', 'product_id'], how='left')

print("PURCHASE ORDER ROWS:", len(po_rows))
print(po_rows[['product_id', 'branch_id', 'branch_id_order', 'reference_id', 'movement_date',
               'received_date', 'quantity', 'order_line_qty', 'po_status']].to_string())
print()
print(po_rows['po_status'].value_counts())

so_rows = last_rows[last_rows['reference_type'] == 'SO'].copy()
so_rows = so_rows.merge(so_h[['so_id', 'branch_id', 'order_date', 'delivery_date', 'order_status']],
                        left_on='reference_id', right_on='so_id', how='left',
                        suffixes=('', '_order'))

print()
print("SALES ORDER ROWS:", len(so_rows))
print(so_rows[['product_id', 'branch_id', 'branch_id_order', 'reference_id', 'movement_date',
               'delivery_date', 'quantity', 'order_status']].to_string())

PURCHASE ORDER ROWS: 16
   product_id branch_id branch_id_order reference_id movement_date received_date  quantity  order_line_qty po_status
0        P002    CHN001          CHN001    PO-148144    2025-01-20    2025-01-20       297             378  Received
1        P004    PUN001          PUN001    PO-541972    2025-01-18    2025-01-18       165             359  Received
2        P008    AHM001          AHM001    PO-625015    2025-01-24    2025-01-24        36              36  Received
3        P010    AHM001          AHM001    PO-327036    2025-01-24    2025-01-24       142             189  Received
4        P010    AHM001          AHM001    PO-625015    2025-01-24    2025-01-24       258             466  Received
5        P021    DEL001          DEL001    PO-167534    2025-01-17    2025-01-17       205             306  Received
6        P023    HYD001          HYD001    PO-817261    2025-01-15    2025-01-15       183             183  Received
7        P025    HYD001          HYD001 

In [26]:
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')

rec = po_l.merge(po_h[['po_id', 'branch_id', 'po_status']], on='po_id', how='left')
rec = rec[rec['po_status'] == 'Received']
rec = rec.groupby(['product_id', 'branch_id'], as_index=False)['quantity'].sum()
rec = rec.rename(columns={'quantity': 'received_units'})

dlv = so_l.merge(so_h[['so_id', 'branch_id', 'order_status']], on='so_id', how='left')
dlv = dlv[dlv['order_status'] == 'Delivered']
dlv = dlv.groupby(['product_id', 'branch_id'], as_index=False)['quantity'].sum()
dlv = dlv.rename(columns={'quantity': 'delivered_units'})

adj = led[led['movement_type'] == 'ADJUSTMENT'].groupby(['product_id', 'branch_id'], as_index=False)['signed_qty'].sum()
adj = adj.rename(columns={'signed_qty': 'adjustment_net'})

ob = inv[['product_id', 'branch_id', 'opening_stock']]
ob = ob.merge(rec, on=['product_id', 'branch_id'], how='left')
ob = ob.merge(dlv, on=['product_id', 'branch_id'], how='left')
ob = ob.merge(adj, on=['product_id', 'branch_id'], how='left').fillna(0)
ob['order_line_stock'] = ob['opening_stock'] + ob['received_units'] - ob['delivered_units'] + ob['adjustment_net']

cmp = ob.merge(rebuilt[['product_id', 'branch_id', 'reconstructed_stock']], on=['product_id', 'branch_id'])
cmp['gap'] = cmp['order_line_stock'] - cmp['reconstructed_stock']

print("Pairs:", len(cmp))
print("Units received (Received POs):", ob['received_units'].sum())
print("Units delivered (Delivered SOs):", ob['delivered_units'].sum())
print("Order-line stock, total:", cmp['order_line_stock'].sum())
print("Ledger-based stock, total:", cmp['reconstructed_stock'].sum())
print("Gap, total:", cmp['gap'].sum())
print("Pairs with order-line stock below zero:", (cmp['order_line_stock'] < 0).sum())
print()
print(cmp['order_line_stock'].describe())
print()
print(cmp['gap'].describe())

Pairs: 180
Units received (Received POs): 22387123
Units delivered (Delivered SOs): 1234939
Order-line stock, total: 21185215
Ledger-based stock, total: 19305994
Gap, total: 1879221
Pairs with order-line stock below zero: 0

count       180.000000
mean     117695.638889
std        5367.487241
min       98642.000000
25%      113562.250000
50%      117900.500000
75%      121422.500000
max      132967.000000
Name: order_line_stock, dtype: float64

count      180.000000
mean     10440.116667
std       1533.925746
min       6226.000000
25%       9361.250000
50%      10458.000000
75%      11409.000000
max      15127.000000
Name: gap, dtype: float64


In [27]:
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
po_status = po_h.set_index('po_id')['po_status']
so_status = so_h.set_index('so_id')['order_status']

led['order_status'] = np.where(led['reference_type'] == 'PO', led['reference_id'].map(po_status),
                        np.where(led['reference_type'] == 'SO', led['reference_id'].map(so_status), 'ADJ'))

print("Ledger rows and units by movement type and order status:")
tab = led.groupby(['movement_type', 'order_status'], dropna=False)['quantity'].agg(['count', 'sum'])
print(tab)

in_received = led[(led['movement_type'] == 'IN') & (led['order_status'] == 'Received')]['quantity'].sum()
in_cancelled = led[(led['movement_type'] == 'IN') & (led['order_status'] == 'Cancelled')]['quantity'].sum()
out_delivered = led[(led['movement_type'] == 'OUT') & (led['order_status'] == 'Delivered')]['quantity'].sum()
out_cancelled = led[(led['movement_type'] == 'OUT') & (led['order_status'] == 'Cancelled')]['quantity'].sum()

missing_in = ob['received_units'].sum() - in_received
missing_out = ob['delivered_units'].sum() - out_delivered

print()
print("Units on Received POs missing from the ledger:", missing_in)
print("Units on Delivered SOs missing from the ledger:", missing_out)
print("Ledger IN from Cancelled POs:", in_cancelled)
print("Ledger OUT from Cancelled SOs:", out_cancelled)
print()
print("Explained gap:", missing_in - missing_out - in_cancelled + out_cancelled)
print("Actual gap:", cmp['gap'].sum())

Ledger rows and units by movement type and order status:
                             count       sum
movement_type order_status                  
ADJUSTMENT    ADJ             2454     66990
IN            Cancelled       1542    248556
              Received      126069  20149239
OUT           Cancelled       1275     13234
              Delivered     105890   1111598

Units on Received POs missing from the ledger: 2237884
Units on Delivered SOs missing from the ledger: 123341
Ledger IN from Cancelled POs: 248556
Ledger OUT from Cancelled SOs: 13234

Explained gap: 1879221
Actual gap: 1879221


In [28]:
checks = {
    'stock_ledger': (ledger, ['product_id', 'branch_id', 'movement_type', 'movement_date',
                              'quantity', 'reference_id', 'running_balance']),
    'inventory_master': (inv, ['product_id', 'branch_id', 'opening_stock', 'max_stock', 'current_stock']),
    'purchase_orders_lines': (po_l, ['po_id', 'product_id', 'quantity']),
    'sales_orders_lines': (so_l, ['so_id', 'product_id', 'quantity']),
    'purchase_orders_header': (po_h, ['po_id', 'branch_id', 'po_status']),
    'sales_orders_header': (so_h, ['so_id', 'branch_id', 'order_status']),
}

print("CHECK 1: empty values, zeros and negatives")
for name, (df, cols) in checks.items():
    print(name, "| rows:", len(df))
    for c in cols:
        line = "   " + c + ": empty = " + str(df[c].isnull().sum())
        if pd.api.types.is_numeric_dtype(df[c]):
            line += " | zeros = " + str((df[c] == 0).sum()) + " | negatives = " + str((df[c] < 0).sum())
        print(line)

print()
print("CHECK 2: every pair has movements and order lines")
cnt = led.groupby(['product_id', 'branch_id', 'movement_type']).size().unstack(fill_value=0)
print("Pairs counted:", len(cnt))
print("Pairs with no IN rows:", (cnt['IN'] == 0).sum())
print("Pairs with no OUT rows:", (cnt['OUT'] == 0).sum())
print("Pairs with no received units:", (ob['received_units'] == 0).sum())
print("Pairs with no delivered units:", (ob['delivered_units'] == 0).sum())

CHECK 1: empty values, zeros and negatives
stock_ledger | rows: 237230
   product_id: empty = 0
   branch_id: empty = 0
   movement_type: empty = 0
   movement_date: empty = 0
   quantity: empty = 0 | zeros = 0 | negatives = 0
   reference_id: empty = 0
   running_balance: empty = 0 | zeros = 0 | negatives = 0
inventory_master | rows: 180
   product_id: empty = 0
   branch_id: empty = 0
   opening_stock: empty = 0 | zeros = 0 | negatives = 0
   max_stock: empty = 0 | zeros = 0 | negatives = 0
   current_stock: empty = 0 | zeros = 0 | negatives = 0
purchase_orders_lines | rows: 155495
   po_id: empty = 0
   product_id: empty = 0
   quantity: empty = 0 | zeros = 0 | negatives = 0
sales_orders_lines | rows: 130402
   so_id: empty = 0
   product_id: empty = 0
   quantity: empty = 0 | zeros = 0 | negatives = 0
purchase_orders_header | rows: 24000
   po_id: empty = 0
   branch_id: empty = 0
   po_status: empty = 0
sales_orders_header | rows: 20000
   so_id: empty = 0
   branch_id: empty = 0


In [29]:
key = ['product_id', 'branch_id']

ledger_stock = ledger.groupby(key)['running_balance'].last().rename('ledger_stock')
in_u  = ledger[ledger.movement_type == 'IN'].groupby(key)['quantity'].sum().rename('in_units')
out_u = ledger[ledger.movement_type == 'OUT'].groupby(key)['quantity'].sum().rename('out_units')

po = po_l.merge(po_h[['po_id', 'branch_id', 'po_status']], on='po_id')
so = so_l.merge(so_h[['so_id', 'branch_id', 'order_status']], on='so_id')
received  = po[po.po_status == 'Received'].groupby(key)['quantity'].sum().rename('received')
delivered = so[so.order_status == 'Delivered'].groupby(key)['quantity'].sum().rename('delivered')

t = inv[key + ['opening_stock', 'max_stock']].merge(ledger_stock, on=key) \
      .merge(in_u, on=key).merge(out_u, on=key) \
      .merge(received, on=key).merge(delivered, on=key)

t['net_adj'] = t.ledger_stock - t.opening_stock - t.in_units + t.out_units
t['order_line_stock'] = t.opening_stock + t.received - t.delivered + t.net_adj

print("Pairs:", len(t))
print("Total net adjustments (expect 62):", t.net_adj.sum())
print("Total ledger stock (expect 19,305,994):", t.ledger_stock.sum())
print("Total order-line stock (expect 21,185,215):", t.order_line_stock.sum())

Pairs: 180
Total net adjustments (expect 62): 62
Total ledger stock (expect 19,305,994): 19305994
Total order-line stock (expect 21,185,215): 21185215


In [30]:
t['ratio_ledger'] = t.ledger_stock / t.max_stock
t['ratio_orderline'] = t.order_line_stock / t.max_stock

print("Times above max_stock, ledger-based:")
print(t.ratio_ledger.describe().round(1))
print("\nTimes above max_stock, order-line:")
print(t.ratio_orderline.describe().round(1))

print("\nPairs at or below max_stock, ledger-based:", int((t.ledger_stock <= t.max_stock).sum()))
print("Pairs at or below max_stock, order-line:", int((t.order_line_stock <= t.max_stock).sum()))

Times above max_stock, ledger-based:
count    180.0
mean     385.1
std      155.8
min      177.4
25%      259.2
50%      349.4
75%      477.5
max      858.1
Name: ratio_ledger, dtype: float64

Times above max_stock, order-line:
count    180.0
mean     422.6
std      170.6
min      193.2
25%      280.9
50%      385.6
75%      525.3
max      929.2
Name: ratio_orderline, dtype: float64

Pairs at or below max_stock, ledger-based: 0
Pairs at or below max_stock, order-line: 0


In [31]:
for col in ['ledger_stock', 'order_line_stock']:
    q1, q3 = t[col].quantile(0.25), t[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_low  = int((t[col] < low).sum())
    n_high = int((t[col] > high).sum())
    print(f"{col}: q1 = {q1:,.0f}, q3 = {q3:,.0f}, normal range {low:,.0f} to {high:,.0f}")
    print(f"   pairs below range = {n_low}, pairs above range = {n_high}\n")

ledger_stock: q1 = 103,488, q3 = 110,780, normal range 92,550 to 121,718
   pairs below range = 1, pairs above range = 0

order_line_stock: q1 = 113,562, q3 = 121,422, normal range 101,772 to 133,213
   pairs below range = 1, pairs above range = 0



In [32]:
show = ['product_id', 'branch_id', 'max_stock', 'ledger_stock', 'order_line_stock', 'ratio_orderline']

print("5 smallest pairs (order-line stock):")
print(t.nsmallest(5, 'order_line_stock')[show].round(1).to_string(index=False))

print("\n5 largest pairs (order-line stock):")
print(t.nlargest(5, 'order_line_stock')[show].round(1).to_string(index=False))

print("\n5 pairs with the highest ratio to max_stock:")
print(t.nlargest(5, 'ratio_orderline')[show].round(1).to_string(index=False))

print("\nmax_stock across the 180 pairs: min", t.max_stock.min(), "| max", t.max_stock.max())

5 smallest pairs (order-line stock):
product_id branch_id  max_stock  ledger_stock  order_line_stock  ratio_orderline
      P019    KOL001        141         88853             98642            699.6
      P015    KOL001        508         96547            106571            209.8
      P018    KOL001        284         96607            106960            376.6
      P013    AHM001        150         98687            107342            715.6
      P016    KOL001        277         97829            107407            387.8

5 largest pairs (order-line stock):
product_id branch_id  max_stock  ledger_stock  order_line_stock  ratio_orderline
      P002    HYD001        383        121013            132967            347.2
      P012    DEL001        489        119923            129402            264.6
      P003    PUN001        359        118139            129186            359.8
      P017    HYD001        172        117058            128477            747.0
      P007    DEL001        383    

In [33]:
l_dates = pd.to_datetime(ledger['movement_date'])
r_dates = pd.to_datetime(po_h['received_date'])
d_dates = pd.to_datetime(so_h['delivery_date'])

print("Ledger movement_date:", l_dates.min().date(), "to", l_dates.max().date())
print("PO received_date:    ", r_dates.min().date(), "to", r_dates.max().date())
print("SO delivery_date:    ", d_dates.min().date(), "to", d_dates.max().date())

print("\nreceived_date empty:", po_h['received_date'].isna().sum())
print(po_h[po_h['received_date'].isna()]['po_status'].value_counts())

print("\ndelivery_date empty:", so_h['delivery_date'].isna().sum())
print(so_h[so_h['delivery_date'].isna()]['order_status'].value_counts())

print("\nreference_type empty:", ledger['reference_type'].isna().sum())
print(ledger['reference_type'].value_counts(dropna=False))

Ledger movement_date: 2019-01-01 to 2025-01-28
PO received_date:     2019-01-11 to 2025-01-28
SO delivery_date:     2019-01-02 to 2025-01-13

received_date empty: 2370
po_status
Cancelled    2370
Name: count, dtype: int64

delivery_date empty: 0
Series([], Name: count, dtype: int64)

reference_type empty: 0
reference_type
PO     127611
SO     107165
ADJ      2454
Name: count, dtype: int64


In [34]:
ld = pd.to_datetime(ledger['movement_date'])

late = ledger[ld > '2025-01-13']
print("Ledger rows after 13 Jan 2025:", len(late))
print(late.groupby(['reference_type', 'movement_type']).size())

print("\nLast ledger date for each reference_type:")
print(ledger.assign(d=ld).groupby('reference_type')['d'].max().dt.date)

Ledger rows after 13 Jan 2025: 347
reference_type  movement_type
PO              IN               347
dtype: int64

Last ledger date for each reference_type:
reference_type
ADJ    2024-12-29
PO     2025-01-28
SO     2025-01-13
Name: d, dtype: object


In [35]:
pairs = [('P019', 'KOL001'), ('P002', 'HYD001'), ('P025', 'HYD001')]

for p, b in pairs:
    L = ledger[(ledger.product_id == p) & (ledger.branch_id == b)]
    opening = inv[(inv.product_id == p) & (inv.branch_id == b)]['opening_stock'].iloc[0]
    in_q  = L[L.movement_type == 'IN']['quantity'].sum()
    out_q = L[L.movement_type == 'OUT']['quantity'].sum()

    adj = L[L.reference_type == 'ADJ']
    adj_net = adj[adj.movement_type == 'IN']['quantity'].sum() - adj[adj.movement_type == 'OUT']['quantity'].sum()

    rec = po[(po.product_id == p) & (po.branch_id == b) & (po.po_status == 'Received')]['quantity'].sum()
    dlv = so[(so.product_id == p) & (so.branch_id == b) & (so.order_status == 'Delivered')]['quantity'].sum()

    row = t[(t.product_id == p) & (t.branch_id == b)].iloc[0]

    print(f"--- {p} {b} ---")
    print("Opening:", opening, "| IN:", in_q, "| OUT:", out_q)
    print("By hand (opening + IN - OUT):", opening + in_q - out_q,
          "| last running_balance:", L['running_balance'].iloc[-1],
          "| table ledger_stock:", row.ledger_stock)
    print("Adjustments by hand:", adj_net, "| table net_adj:", row.net_adj)
    print("Received:", rec, "| Delivered:", dlv)
    print("Order-line by hand:", opening + rec - dlv + adj_net,
          "| table order_line_stock:", row.order_line_stock, "\n")

--- P019 KOL001 ---
Opening: 92 | IN: 95724 | OUT: 7135
By hand (opening + IN - OUT): 88681 | last running_balance: 88853 | table ledger_stock: 88853
Adjustments by hand: 0 | table net_adj: 172
Received: 106282 | Delivered: 7904
Order-line by hand: 98470 | table order_line_stock: 98642 

--- P002 HYD001 ---
Opening: 219 | IN: 125818 | OUT: 4958
By hand (opening + IN - OUT): 121079 | last running_balance: 121013 | table ledger_stock: 121013
Adjustments by hand: 0 | table net_adj: -66
Received: 138227 | Delivered: 5413
Order-line by hand: 133033 | table order_line_stock: 132967 

--- P025 HYD001 ---
Opening: 83 | IN: 119686 | OUT: 5512
By hand (opening + IN - OUT): 114257 | last running_balance: 114128 | table ledger_stock: 114128
Adjustments by hand: 0 | table net_adj: -129
Received: 129609 | Delivered: 5977
Order-line by hand: 123715 | table order_line_stock: 123586 



In [36]:
print("How each reference_type is marked in movement_type:")
print(ledger.groupby(['reference_type', 'movement_type']).size(), "\n")

for p, b in [('P019', 'KOL001'), ('P002', 'HYD001'), ('P025', 'HYD001')]:
    L = ledger[(ledger.product_id == p) & (ledger.branch_id == b)].copy()
    opening = inv[(inv.product_id == p) & (inv.branch_id == b)]['opening_stock'].iloc[0]
    L['before'] = L['running_balance'].shift(1)
    L.iloc[0, L.columns.get_loc('before')] = opening
    L['change'] = L['running_balance'] - L['before']

    adj = L[L.reference_type == 'ADJ']
    other = L[L.reference_type != 'ADJ']
    table_adj = t[(t.product_id == p) & (t.branch_id == b)]['net_adj'].iloc[0]

    print(f"--- {p} {b} ---")
    print("ADJ rows:", len(adj), "| total change from ADJ rows:", adj['change'].sum(), "| table net_adj:", table_adj)
    print("Rows where change differs from quantity (non-ADJ):",
          int((other['change'].abs() != other['quantity']).sum()))
    print(adj[['movement_date', 'movement_type', 'quantity', 'before', 'running_balance', 'change']].to_string(index=False), "\n")

How each reference_type is marked in movement_type:
reference_type  movement_type
ADJ             ADJUSTMENT         2454
PO              IN               127611
SO              OUT              107165
dtype: int64 

--- P019 KOL001 ---
ADJ rows: 11 | total change from ADJ rows: 172.0 | table net_adj: 172
Rows where change differs from quantity (non-ADJ): 0
movement_date movement_type  quantity  before  running_balance  change
   2019-02-11    ADJUSTMENT        24  1668.0             1644   -24.0
   2019-11-15    ADJUSTMENT         8 12902.0            12910     8.0
   2021-03-20    ADJUSTMENT        29 32492.0            32521    29.0
   2021-06-02    ADJUSTMENT        46 35519.0            35565    46.0
   2021-08-09    ADJUSTMENT        12 37126.0            37138    12.0
   2021-08-22    ADJUSTMENT        48 38124.0            38076   -48.0
   2022-05-11    ADJUSTMENT        47 50927.0            50974    47.0
   2023-06-22    ADJUSTMENT        39 67645.0            67684    39.0
 